# Lecture 06 companion notebook — R: files, data frames and figures

**BIOE 610 Computational Tools · Week 6**

Every example from Lecture 06 as cells you can run, change and break — reading and writing files, drawing random samples, base graphics, and data frames. There is a matching Python notebook (`Lecture06_Python_dataframes_figures.ipynb`) with the same sections in the same order — keep them open side by side to compare. Today needs **no packages at all**: everything is base R.

**How to use this notebook in Positron**

1. Open the file — Positron shows it as a notebook automatically
2. Pick the **R kernel** (top right). If no R kernel is offered, run this once in an R console, then reopen:
   ```r
   install.packages("IRkernel")
   IRkernel::installspec()
   ```
3. Run a cell with **Shift-Enter**. Run them in order the first time; then go back and experiment — *change the numbers, break things on purpose, read the errors*.

# Part 1 · Reading and writing files

## 1 · Reading tabular data

`read.csv()` for comma-separated files, `read.delim()` for tab-separated ones. Both read straight from a URL, so this notebook works wherever you saved it. `check.names = FALSE` keeps the column name `Geno&Micro` exactly as it is in the file.

In [ ]:
stickle <- read.delim("https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv",
                      check.names = FALSE)
dim(stickle)          # rows, columns
stickle[1:3, 1:5]     # first rows, first columns

In [ ]:
# The two questions every reader function asks: where are the column
# names (header) and what separates the values (sep)?
# Without check.names, R "fixes" awkward names: Geno&Micro becomes Geno.Micro.
# Try it - read just one row and compare the 4th column name:
names(read.delim("https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv", nrows = 1))[4]
names(stickle)[4]     # what we kept with check.names = FALSE

## 2 · Writing tabular data

Write **derived** tables into `results/` — never overwrite anything in `data/` (raw data is read-only). `row.names = FALSE`, or R sneaks an extra unnamed column of row numbers into your file.

In [ ]:
genes  <- grep("^Gene", names(stickle))              # which columns are genes?
totals <- data.frame(Individual = stickle$Individual,
                     Total      = rowSums(stickle[, genes]))
head(totals, 3)

In [ ]:
dir.create("results", showWarnings = FALSE)
write.csv(totals, "results/totals.csv", row.names = FALSE)
write.table(totals, "results/totals.tsv", sep = "\t",
            row.names = FALSE, quote = FALSE)
list.files("results")

## 3 · Saving figures to files

R's pattern is *open a file device → plot → `dev.off()`*. Forget `dev.off()` and the file is empty or locked. Vector formats (`.svg`, `.pdf`) scale forever — use them for papers; raster (`.png`) needs a resolution (`res = 300`).

In [ ]:
svg("results/first_figure.svg", width = 7, height = 5)   # open the device
boxplot(totals$Total, ylab = "Total reads per fish")      # draw into it
dev.off()                                                 # close it - ALWAYS

png("results/first_figure.png", width = 7, height = 5, units = "in", res = 300)
boxplot(totals$Total, ylab = "Total reads per fish")
dev.off()

list.files("results")

# Part 2 · Drawing samples from distributions

## 4 · Drawing random samples

The `r*` family — `rnorm()`, `runif()`, `rpois()`, `rbinom()` — draws from named distributions; `sample()` draws from anything you give it. `set.seed()` makes the "random" numbers reproducible.

In [ ]:
set.seed(2026)
x <- rnorm(10000, mean = 0, sd = 10)
y <- sample(1:10000, 10000, replace = TRUE)
plot(x, y, pch = 16, cex = 0.3, col = adjustcolor("#31688e", 0.4))

In [ ]:
# Your turn within a cell: change mean and sd and re-run.
# What does sd = 1 vs sd = 50 do to the cloud?
hist(rnorm(10000, mean = 0, sd = 10), breaks = 40)

## 5 · Histograms and the honest density overlay

`freq = FALSE` puts the bars on the **density** scale, so the theoretical curve `dnorm()` drops straight on top — no magic scaling constants. `curve(..., add = TRUE)` is a *low-level* call: it draws onto the open figure.

In [ ]:
x <- rnorm(1000, mean = 0, sd = 100)
hist(x, breaks = 30, freq = FALSE, col = "grey90", border = "white")
curve(dnorm(x, mean = 0, sd = 100), add = TRUE, col = "#440154", lwd = 2)

## 6 · Four distributions at once — and multi-panel figures

`par(mfrow = c(2, 2))` splits the plotting device into a grid; each high-level call fills the next panel. Four workhorses: **normal** (measurements), **uniform** (anything equally likely), **Poisson** (counts), **binomial** (successes out of *n* tries).

In [ ]:
set.seed(2026); n <- 2000
par(mfrow = c(2, 2), mar = c(4, 4, 2, 1))
hist(rnorm(n, 50, 10),   breaks = 40, col = "#440154", border = "white",
     main = "Normal(50, 10)",    xlab = "value")
hist(runif(n, 0, 100),   breaks = 40, col = "#31688e", border = "white",
     main = "Uniform(0, 100)",   xlab = "value")
hist(rpois(n, 5),        breaks = seq(-0.5, 20.5, 1), col = "#35b779",
     border = "white", main = "Poisson(5)",        xlab = "count")
hist(rbinom(n, 20, 0.3), breaks = seq(-0.5, 20.5, 1), col = "#fde725",
     border = "white", main = "Binomial(20, 0.3)", xlab = "successes")
par(mfrow = c(1, 1))     # reset, or every later plot lands in the grid

In [ ]:
# Change n, the parameters and breaks until each panel surprises you.
# What happens to Poisson(5) as you raise lambda to 50?
hist(rpois(2000, 50), breaks = 40, col = "#35b779", border = "white")

# Part 3 · Base graphics and data frames

## 7 · Base graphics vocabulary

**High-level** functions start a figure (`plot()`, `hist()`, `boxplot()`); **low-level** functions add to the open one (`points()`, `lines()`, `curve(add = TRUE)`, `legend()`). Arguments do the styling: `type=`, `col=`, `pch=`, `lwd=`, `xlab=` / `ylab=` / `main=`.

In [ ]:
seq_1 <- seq(0, 10, by = 0.1)
plot(seq_1, sin(seq_1), type = "l", lwd = 2, col = "#440154",
     xlab = "x", ylab = "sin(x)", main = "plot(), then points(), then legend()")
points(seq_1[seq(1, 101, by = 10)], sin(seq_1)[seq(1, 101, by = 10)],
       pch = 16, col = "#35b779")
legend("topright", legend = c("line", "every 10th point"),
       col = c("#440154", "#35b779"), lty = c(1, NA), pch = c(NA, 16), bty = "n")

## 8 · Creating a data frame by hand

Vectors of equal length become columns. A data frame is a *list of equal-length vectors* — one name per column, one type per column.

In [ ]:
hydrogel_concentration <- factor(c("low", "high", "high", "high",
                                   "medium", "medium", "medium", "low"))
compression  <- c(3.4, 3.4, 8.4, 3, 5.6, 8.1, 8.3, 4.5)
conductivity <- c(0, 9.2, 3.8, 5, 5.6, 4.1, 7.1, 5.3)

mydata <- data.frame(hydrogel_concentration, compression, conductivity)
row.names(mydata) <- paste0("Sample_", 1:8)
mydata

## 9 · The stickleback table as a data frame

First questions for any new table: how big (`dim`), what's in it (`head`, `str`), how do the groups break down (`table`), what does one column look like (`summary`).

In [ ]:
dim(stickle)
table(stickle$Genotype, stickle$Microbiota)
summary(stickle$Gene1)

## 10 · Summary statistics by group

`aggregate()`'s formula — *response `~` groups* — is exactly the formula `boxplot()` uses in the next section. The tidyverse's `group_by() |> summarize()` generalizes this next week.

In [ ]:
aggregate(Gene1 ~ Genotype + Microbiota, data = stickle, FUN = median)

In [ ]:
sapply(stickle[, c("Gene1", "Gene2", "Gene3")], mean)

## 11 · A first real figure

In [ ]:
boxplot(Gene1 ~ Genotype + Microbiota, data = stickle,
        col = c("steelblue", "tomato", "steelblue", "tomato"),
        ylab = "Gene1 read count", las = 2)

## 12 · The same figure, dressed for a lab meeting

Every improvement is one argument or one extra call — colors, names, a legend, real labels. It works, but it is manual bookkeeping; ggplot2 derives all of this *from the data* next week.

In [ ]:
boxplot(Gene1 ~ Microbiota + Genotype, data = stickle,
        col = c("#44015499", "#35b77999"),
        names = c("conv\nA", "mono\nA", "conv\nB", "mono\nB"),
        las = 1, xlab = "", ylab = "Gene1 read count",
        main = "Stickleback gut RNA-seq: Gene1 by group")
legend("topright", fill = c("#44015499", "#35b77999"), title = "Microbiota",
       legend = c("conventional", "mono"), bty = "n")

## Your turn

1. Pick another gene (`Gene2` … `Gene600`) and redo sections 10–12 for it. Which gene shows the biggest group difference you can find?
2. Change the gallery in section 6: new parameters, new `breaks`, or swap a panel for `rexp(n, rate = 0.2)`.
3. Put it in a script: `scripts/summarize_stickleback.R` that reads the data, writes a per-group median CSV into `results/`, and saves the gallery as `results/distributions.svg` *and* a 300-dpi PNG. A working version is on the Examples page — but only look after your own runs.